# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR²) Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library, referencing all entities by their Croissant `@id` fields as required.

### Dataset Source
The dataset is described by a Croissant JSON-LD schema available at the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs. We will list all record sets, their fields, and show how to access their data objects by `@id`.

In [ ]:
# List all record sets in the dataset with their @id
record_sets = list(dataset.record_sets)
print(f"Total record sets found: {len(record_sets)}\n")

for rs in record_sets:
    print(f"RecordSet name: {rs.get('name', '[no name]')}")
    print(f"  @id: {rs['@id']}")
    # List all fields in the record set
    if 'field' in rs and isinstance(rs['field'], list):
        print("  Fields:")
        for fld in rs['field']:
            fld_id = fld['@id'] if isinstance(fld, dict) else fld
            print(f"    - {fld_id}")
    elif 'field' in rs:
        print(f"  Field: {rs['field']['@id'] if isinstance(rs['field'], dict) else rs['field']}")
    print()

# Show a few example records from each record set by @id
for rs in record_sets:
    print(f"Sample records for record set @id: {rs['@id']}")
    try:
        for i, rec in enumerate(dataset.records(record_set=rs['@id'])):
            print(rec)
            if i >= 1:
                break
    except Exception as e:
        print(f"Could not read records: {e}")
    print("\n------\n")

## 3. Data Extraction
Load data from all record sets into Pandas DataFrames for analysis. Here, every record set is referenced by its `@id` field.

In [ ]:
# Extract data from all record sets
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"RecordSet '@id': {record_set_id} => Loaded shape: {dataframes[record_set_id].shape}")
    else:
        print(f"RecordSet '@id': {record_set_id} => No records loaded.")

# If only one record set, display its columns and a sample
if dataframes:
    main_rs = max(dataframes, key=lambda k: dataframes[k].shape[0])
    print(f"\nColumns in main record set (@id: {main_rs}):")
    print(dataframes[main_rs].columns.tolist())
    dataframes[main_rs].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. We'll use entity `@id`s to reference all attributes.

In [ ]:
# Identify a record set with numeric fields for analysis
# (We'll display candidate numeric fields)
import numpy as np

main_rs = max(dataframes, key=lambda k: dataframes[k].shape[0]) if dataframes else None
if main_rs is not None:
    df = dataframes[main_rs]
    # Heuristically find numeric-looking fields
    numeric_field_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    print(f"Numeric field candidates (@id): {numeric_field_candidates}")

    # For demonstration, pick the first numeric field (if available)
    if numeric_field_candidates:
        numeric_field_id = numeric_field_candidates[0]
        print(f"\nAnalyzing numeric field with @id: {numeric_field_id}\n")

        threshold = df[numeric_field_id].quantile(0.75) if len(df) > 0 else 10
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by first non-numeric field if available
        group_field_candidates = [col for col in df.columns if col != numeric_field_id and not pd.api.types.is_numeric_dtype(df[col])]
        if group_field_candidates:
            group_field = group_field_candidates[0]
            if group_field in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
                print(f"\nGrouped mean of {numeric_field_id} by {group_field} (@id):")
                print(grouped_df.head())
        else:
            print("No non-numeric fields available for grouping.")
    else:
        print("No numeric fields found in the main DataFrame.")
else:
    print("No record set with data was found.")

## 5. Visualization
Visualize distributions or relationships using the selected fields by their @id. Example: plot the distribution of the numeric field, and a boxplot grouped by the selected group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only proceed if numeric_field_id and group_field are found
if 'numeric_field_id' in locals() and numeric_field_candidates and (len(filtered_df) > 0):
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.show()

    if 'group_field' in locals():
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field_id])
        plt.title(f"Boxplot of '{numeric_field_id}' grouped by '{group_field}' (@id)")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to use the `mlcroissant` library to load, inspect, and analyze a FAIR-compliant biomedical dataset described by a Croissant schema. All dataset entities—including record sets, fields, and columns—were referenced by their `@id`.

We explored the structure, extracted tabular records into DataFrames, applied basic filtering and normalization operations, and visualized major data distributions. This approach provides a reproducible and standardized pipeline for biomedical dataset analysis using open metadata standards.